## Main Assignment Machine Learning: Ela, Emma, Florence, Jelle 28-03-2026

Importing packages 

In [12]:
import pandas as pd
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn import datasets as ds
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import decomposition

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn.metrics as sklm
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import plot_tree
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report, average_precision_score, roc_auc_score

from scipy.stats import ttest_ind

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

scores_dict = {}


Load data 

In [13]:
with zipfile.ZipFile("ecg_data.zip","r") as zip_ref:
    zip_ref.extractall("ecg_data")

def load_data():
    this_directory = os.getcwd()
    data = pd.read_csv(os.path.join(this_directory, 'ecg_data/ecg_data.csv'), index_col=0)
    return data

raw_data = load_data()

Data description

In [14]:
print(f'The number of samples: {len(raw_data.index)}')
print(f'The number of columns: {len(raw_data.columns)}')

print(f'The number of NaN values in the entire dataframe: {raw_data.isnull().sum().sum()}')
print(f'The number of samples with label 0: {len(raw_data[raw_data["label"] == 0])}')
print(f'The number of samples with label 1: {len(raw_data[raw_data["label"] == 1])}')
print(f'The percentage of samples with label 0 is thus {len(raw_data[raw_data["label"] == 0])/len(raw_data.index)*100:.2f}%', 
      f'and the percentage with label 1 {len(raw_data[raw_data["label"] == 1])/len(raw_data.index)*100:.2f}%')

# print(raw_data.groupby('label').count())
# print(raw_data.groupby('label').mean())
# print(raw_data.groupby('label').var())
# print(raw_data.groupby('label').std())


The number of samples: 827
The number of columns: 9001
The number of NaN values in the entire dataframe: 0
The number of samples with label 0: 681
The number of samples with label 1: 146
The percentage of samples with label 0 is thus 82.35% and the percentage with label 1 17.65%


Splitting the data in training and test sets

In [15]:
X = raw_data.drop('label', axis=1)
Y = raw_data['label']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20, random_state=4, stratify=Y)

Preprocessing of data

In [ ]:

class VarianceFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.columns_to_drop_ = [col for col in X.columns if np.var(X[col]) < self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.columns_to_drop_ = [col for col in upper.columns if any(upper[col] > self.threshold)]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class TTestFilter(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=0.05):
        self.alpha = alpha

    def fit(self, X, y):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]

        self.columns_to_drop_ = []
        threshold = self.alpha / X.shape[1]  # Bonferroni

        for col in X.columns:
            group0 = X.loc[y == 0, col]
            group1 = X.loc[y == 1, col]
            _, p_value = ttest_ind(group0, group1, nan_policy="omit")

            if p_value > threshold:
                self.columns_to_drop_.append(col)

        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


pipeline = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ("scaler", RobustScaler())
    ])

pipeline.fit(x_train, y_train)
x_train_preprocessed = pipeline.transform(x_train)
x_test_preprocessed = pipeline.transform(x_test)

#Amount of features after preprocessing 
x_train_var = pipeline.named_steps["variance"].transform(x_train)
x_train_corr = pipeline.named_steps["correlation"].transform(x_train_var)
x_train_t = pipeline.named_steps["ttest"].transform(x_train_corr)

print(f"Start amount features: {x_train.shape[1]}")
print(f"After the variance filter: {x_train_var.shape[1]}")
print(f"After the correlation filter: {x_train_corr.shape[1]}")
print(f"After the t-test filter: {x_train_t.shape[1]}")

Defining the evaluation on test (Precision, Recall, F1, AUC-PR, ROC and plot of AUC-PR)

In [ ]:
from os import name

def evaluate_model_on_test(clf, X_test, y_test, name="Model"):
    '''
        Call function as follows:
        - scores_dict.update(evaluate_model_on_test(...))

        Naming convention for models:
        - LR        -->     Lasso Logistic Regression
        - RF        -->     Random Forest
        - SVM       -->     Support Vector Machine
        - XGB       -->     XGBoost
        - PLS-DA    -->     Partial Least-Squares Discriminant Analysis
        - NN        -->     Neural Network
    '''
    y_pred = clf.predict(X_test)
    if hasattr(clf, 'predict_proba'):
        y_score = clf.predict_proba(X_test)[:, 1]
    else:
        y_score = clf.decision_function(X_test)

    # Calculate metrics
    precision = precision_score(y_test, y_pred, zero_division=1)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_score)
    auc_roc = roc_auc_score(y_test, y_score)

    # Print resultats
    print(f"--- Evaluation: {name} ---")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"F1-Score:   {f1:.4f}")
    print(f"AUC-PR:     {auc_pr:.4f}")
    print(f"ROC-AUC:    {auc_roc:.4f}")
    print("-" * 30)

    return {name: y_score}


Performing Random forest: 

In [ ]:

# y_train en y_test series van maken
if isinstance(y_train, pd.DataFrame):
    if y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    else:
        raise ValueError("y_train moet 1 kolom bevatten.")

if isinstance(y_test, pd.DataFrame):
    if y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]
    else:
        raise ValueError("y_test moet 1 kolom bevatten.")

pipeline_rf = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ("scaler", RobustScaler()),
    ("rf", RandomForestClassifier(random_state=4, class_weight="balanced"))
])


param_dist = {
    "rf__n_estimators": randint(100, 600),
    "rf__max_depth": [None, 5, 10, 20, 30, 50],
    "rf__min_samples_split": randint(2, 20),
    "rf__min_samples_leaf": randint(1, 10),
    "rf__max_features": ["sqrt", "log2", None],
    "rf__bootstrap": [True, False]
}

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)
random_search = RandomizedSearchCV(
    estimator=pipeline_rf,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=kf,
    verbose=0,
    n_jobs=1,
    random_state=4,
    return_train_score=True
)

random_search.fit(x_train, y_train)

print("Beste parameters:")
print(random_search.best_params_)
print(f"\nBeste cross-validated PR-AUC: {random_search.best_score_:.4f}")

best_rf = random_search.best_estimator_

evaluate_model_on_test(
    best_rf,
    x_test,
    y_test,
    name="Optimized Random Forest"
)

Beste parameters:
{'rf__bootstrap': True, 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 8, 'rf__min_samples_split': 8, 'rf__n_estimators': 140}

Beste cross-validated PR-AUC: 0.5702
--- Evaluation: Optimized Random Forest ---
Precision:  0.4167
Recall:     0.3448
F1-Score:   0.3774
AUC-PR:     0.4932
ROC-AUC:    0.7727
------------------------------


{'Optimized Random Forest': array([0.32274403, 0.34058256, 0.32479916, 0.15158401, 0.17398736,
        0.3104582 , 0.35872866, 0.21471847, 0.36034488, 0.25382521,
        0.38533013, 0.04584769, 0.3989813 , 0.09420022, 0.30005858,
        0.10528347, 0.40692298, 0.08930127, 0.12432594, 0.61313915,
        0.30703313, 0.29129572, 0.60923817, 0.16142967, 0.23046264,
        0.19628822, 0.02594157, 0.3049576 , 0.27199155, 0.61269555,
        0.51328908, 0.18608531, 0.20589663, 0.11563813, 0.09961384,
        0.28309812, 0.09580514, 0.30496093, 0.31552133, 0.34623642,
        0.13634808, 0.29696176, 0.09607697, 0.25117524, 0.43238702,
        0.13179344, 0.17923065, 0.1912782 , 0.78056672, 0.62663142,
        0.13127182, 0.08018656, 0.52920297, 0.38535742, 0.29598789,
        0.23374964, 0.20454415, 0.51449818, 0.15735077, 0.13966168,
        0.31355456, 0.11731636, 0.14453304, 0.08980641, 0.9478581 ,
        0.95956034, 0.32741572, 0.4784938 , 0.13068879, 0.2882663 ,
        0.08500622, 0